# Setup workflow environment

In [18]:
using Pkg; Pkg.activate(dirname(Base.current_project()))

import TulipaIO as TIO
import TulipaEnergyModel as TEM
using DuckDB
using DataFrames, CSV

# For Win. system to fix the KaTex parse error in Jupyter Notebook
Base.show(stdout, ::MIME"text/latex", df::DataFrame) = show(stdout, MIME("text/plain"), df)

#= utility functions
    - print_annual_total_prod(DBconnection, years...)
    - inv_cost, fom_cost, var_cost = objective_terms_value(TulipaProblem, DBconnection)
    - annual_flows_between_assets(DBconnection, from_asset_term, to_asset_term)
=#
include("../src/util_analysis.jl");

  Activating project at `c:\ModellingRepos\VintageDemo`


# Multi-year investment model instances

- Milestone years: 2030, 2040 and 2050.
- The system has 30GW initial wind capacity built in 2025, the model can choose to invest in wind in three milestone years: 2030, 2040, 2050.
- unused tables: `flow-both.csv`, `flows-profiles.csv`

In [ ]:
# Config whether wind turbine is decommissionable
const DECOMM = true
DECOMM && begin
    wind_turbine_name_patterns = Dict("1-no-vintage" => "'wind%'", "2-vintage-standard" => "'wind_0%'", "3-vintage-compact" => "'wind%'")

    parameter_table_dir = "./model-instance-Tulipa/config-instance/decommission"
    # compulsory parameter `decommissionable`: the .csv just serves to show the diff
    df_diff_asset_both = CSV.read(joinpath(parameter_table_dir, "asset-both_decommission.csv"), DataFrame)
    # optional parameter `technical_lifetime`: the .csv may contain asset specific value
    df_diff_asset = CSV.read(joinpath(parameter_table_dir, "asset_tech-lifetime.csv"), DataFrame)
end

Row,instance,asset,technical_lifetime
,String31,String7,Int64
1,1-no-vintage,wind,25
2,2-vintage-standard,wind25,25
3,2-vintage-standard,wind30,25
4,2-vintage-standard,wind40,25
5,2-vintage-standard,wind50,25
6,3-vintage-compact,wind,25


## Instance 1. No technology vintage of units

> Note: the `output_dir` must exist/(be created) beforehand 

### 1.1 Build and run the model instance

In [20]:
instance = "1-no-vintage"
# Define and build the input output directories
input_dir = "model-instance-Tulipa/input-$(instance)"
output_dir = joinpath(@__DIR__, "model-instance-Tulipa/output-$(instance)")

# Always build a new result directory
rm(output_dir, force=true, recursive=true) 
mkdir(output_dir);

# Create the connection and prepare input data
connection_no_vintage = DBInterface.connect(DuckDB.DB)
TIO.read_csv_folder(connection_no_vintage, input_dir)
TEM.populate_with_defaults!(connection_no_vintage)

#### 1.1a (Optional) Config decommissioning and technical lifetime

- see `./config-instance/decommission/asset-both_decommission.csv` for value setup

In [21]:
# [optional] update decommissioning configs and technical lifetime
DECOMM && begin
    _connection = connection_no_vintage

    TIO_tbl = "asset_both"
    TIO_tbl_column = "milestone_year"   # no-vintage method only differs per milestone_year
    param_name = "decommissionable"
    map(
        eachrow(filter(r_ins -> r_ins[Symbol("instance")] == instance, df_diff_asset_both))
    ) do df_row
        TIO.update_tbl(
            _connection, TIO_tbl,
            Dict(param_name => df_row[Symbol(param_name)]),
            where_ = "asset = '$(df_row.asset)' AND $(TIO_tbl_column) = '$(df_row[Symbol(TIO_tbl_column)])'"
        )
    end
    # TIO.update_tbl(_connection, TIO_table, 
    #     Dict(param_name => decomm);
    #     where_ = "$(filter_column) LIKE $(wind_turbine_name_patterns[instance])"   # sql 'where' clause
    # )

    TIO_tbl = "asset"
    TIO_tbl_column = "asset"
    param_name = "technical_lifetime"
    map(
        eachrow(filter(r_ins -> r_ins[Symbol("instance")] == instance, df_diff_asset))
    ) do df_row
        TIO.update_tbl(
            _connection, TIO_tbl,
            Dict(param_name => df_row[Symbol(param_name)]),
            where_ = "asset = '$(df_row.asset)'"
        )
    end
end;

# TIO.get_table(_connection, "asset_both")

1-element Vector{Nothing}:
 nothing

In [22]:
# Run the model instance
multiyear_no_vintage = TEM.run_scenario(
    connection_no_vintage;
    optimizer_parameters = Dict("output_flag" => true), # TEM.default_parameters(HiGHS.Optimizer): Dict("output_flag" => false)
    output_folder = output_dir, 
    # model_file_name = joinpath(output_dir, "model.lp"),
    show_log=true,
    # # To record solving time. 
    # # Always run with this option in a new or restarted kernel to ensure the time is dedicated to this solve only.
    # log_file = joinpath(@__DIR__, "model-instance-Tulipa/model-log_$(instance).txt")
)

Running HiGHS 1.15.0 (git hash: 8396001901): Copyright (c) 2026 under MIT licence terms
Includes third-party software components, see THIRD_PARTY_NOTICES.md for full details
MIP has 341640 rows; 315366 cols; 735822 nonzeros; 6 integer variables (0 binary)
Coefficient ranges:
  Matrix  [2e-04, 1e+00]
  Cost    [1e-03, 3e+03]
  Bound   [1e+02, 2e+02]
  RHS     [5e-03, 9e+01]
Presolving model
26277 rows, 140294 cols, 245390 nonzeros 0s
24957 rows, 138778 cols, 237166 nonzeros 6s
Presolve reductions: rows 24957(-316683); columns 138778(-176588); nonzeros 237166(-498656) 

Solving MIP model with:
   24957 rows
   138778 cols (0 binary, 6 integer, 0 implied int., 138772 continuous, 0 domain fixed)
   237166 nonzeros
   Thread count 4 (of 8 threads). Using 1 max workers. Parallel search off

Src: B => Branching; C => Central rounding; F => Feasibility pump; H => Heuristic;
     I => Shifting; J => Feasibility jump; L => Sub-MIP; P => Empty MIP; R => Randomized rounding;
     S => Solve LP; T 

EnergyProblem:
  - Model created!
    - Number of variables: 315366
    - Number of constraints for variable bounds: 315366
    - Number of structural constraints: 341640
  - Model solved!
    - Termination status: OPTIMAL
    - Objective value: 797099.8579018476
    - Objective breakdown:
      - assets_fixed_cost_aggregated_vintage_method: 134906.66285790125
      - assets_fixed_cost_compact_vintage_method: 0.0
      - assets_investment_cost: 255472.35989890975
      - flows_fixed_cost: 0.0
      - flows_investment_cost: 0.0
      - flows_operational_cost: 406720.8351450329
      - storage_assets_energy_fixed_cost: 0.0
      - storage_assets_energy_investment_cost: 0.0
      - units_on_operational_cost: 0.0
      - vintage_flows_operational_cost: 0.0


### 1.2 Key results

#### Capacity

- In this case, we will not be able to differentiate assets built within the same milestone year, they will simply be considered the same as the units built in the milestone year.

In [23]:
function print_capacity_investment(f::Function, DB_conn::DuckDB.DB, decommissionable::Bool)
    # initial wind capacity
    println(
        "Initial wind capacity (GW): ", 
        filter(row -> f(row) && row.initial_units > 0.0, TIO.get_table(DB_conn, "asset_both")[
            :, [:asset, :milestone_year, :commission_year, :decommissionable, :initial_units]
        ]), 
        "\n")
    # invested capacity
    println(
        "Invested wind capacity (GW): # of rows = # of investment variables.\n", 
        filter(f, TIO.get_table(DB_conn, "var_assets_investment")[
            :, [:asset, :milestone_year, :investment_integer, :capacity, :investment_limit, :solution]
        ]), 
        "\n")
    # decommissioned capacity if applicable
    decommissionable && println(
        "Decommissioned wind capacity (GW): # of rows = # of decommissioning variables.\n", 
        filter(f, TIO.get_table(DB_conn, "var_assets_decommission")[
            :, [:asset, :milestone_year, :commission_year, :decommissionable, :initial_units, :investment_integer, :solution]
        ]), 
        "\n")
end

print_capacity_investment(row -> row.asset=="wind", connection_no_vintage, DECOMM)

Initial wind capacity (GW): 3×5 DataFrame
 Row │ asset   milestone_year  commission_year  decommissionable  initial_units 
     │ String  Int32           Int32            Bool              Float64       
─────┼──────────────────────────────────────────────────────────────────────────
   1 │ wind              2030             2030              true           30.0
   2 │ wind              2040             2040              true           30.0
   3 │ wind              2050             2050              true           30.0

Invested wind capacity (GW): # of rows = # of investment variables.
3×6 DataFrame
 Row │ asset   milestone_year  investment_integer  capacity  investment_limit  solution 
     │ String  Int32           Bool                Float64   Float64           Float64  
─────┼──────────────────────────────────────────────────────────────────────────────────
   1 │ wind              2030                true       1.0           107.567     107.0
   2 │ wind              2040        

#### Annual productions & total system cost

In [24]:
print_annual_total_prod(connection_no_vintage, 2030, 2040, 2050)

total_cost = multiyear_no_vintage.objective_value
println("Total system cost: $(round(total_cost/1000, digits=2)) Billion €")

inv_cost, fixed_om_cost, variable_om_cost = objective_terms_value(multiyear_no_vintage, connection_no_vintage)
println(
    "\t investment: $(round(inv_cost/1000, digits=2)) Billion € \n", 
    "\t fixed O&M: $(round(fixed_om_cost/1000, digits=2)) Billion € \n",
    "\t variable O&M: $(round(variable_om_cost/1000, digits=2)) Billion €"
)
println(
    "Total system cost ≈ investment + fixed O&M + variable O&M: ", 
    total_cost ≈ inv_cost + fixed_om_cost + variable_om_cost
)
# @assert total_cost ≈ inv_cost + fixed_om_cost + variable_om_cost

2030s
	 wind prodution: 605.61 TWh p.a.
	 market supply: 207.05 TWh p.a.
2040s
	 wind prodution: 858.42 TWh p.a.
	 market supply: 271.03 TWh p.a.
2050s
	 wind prodution: 844.13 TWh p.a.
	 market supply: 386.1 TWh p.a.
Total system cost: 797.1 Billion €
	 investment: 255.47 Billion € 
	 fixed O&M: 134.91 Billion € 
	 variable O&M: 406.72 Billion €
Total system cost ≈ investment + fixed O&M + variable O&M: true


## Instance 2. Explicit technology vintage of units - standard method

- Difference from **Instance 1**: `asset.csv`, `asset-milestone.csv`, `asset-commission.csv`, `asset-both`, `assets-profiles.csv`, `flow*.csv`

### 2.1 Build and run the model instance

In [25]:
instance = "2-vintage-standard"
# Define and build the input output directories
input_dir = "model-instance-Tulipa/input-$(instance)"
output_dir = joinpath(@__DIR__, "model-instance-Tulipa/output-$(instance)")

# Always build a new result directory
rm(output_dir, force=true, recursive=true) 
mkdir(output_dir);

# Create the connection and prepare input data
connection_vintage_standard = DBInterface.connect(DuckDB.DB)
TIO.read_csv_folder(connection_vintage_standard, input_dir)
TEM.populate_with_defaults!(connection_vintage_standard)

#### 2.1a (Optional) Config decommissioning and technical lifetime

- see `./config-instance/decommission/asset-both_decommission.csv` for value setup
- **Assumption:**
    - assets that are not investable (i.e. 2025 wind vintage) should not be decommissionable either
    - assets should not be decommissionable before and in its commission (vintage) year

In [26]:
DECOMM && begin
    _connection = connection_vintage_standard

    TIO_tbl = "asset_both"
    TIO_tbl_column = "milestone_year"   # standard method distinguishes asset vintages by asset name per milestone_year
    param_name = "decommissionable"
    map(
        eachrow(filter(r_ins -> r_ins[Symbol("instance")] == instance, df_diff_asset_both))
    ) do df_row
        TIO.update_tbl(
            _connection, TIO_tbl,
            Dict(param_name => df_row[Symbol(param_name)]),
            where_ = "asset = '$(df_row.asset)' AND $(TIO_tbl_column) = '$(df_row[Symbol(TIO_tbl_column)])'"
        )
    end

    TIO_tbl = "asset"
    TIO_tbl_column = "asset"
    param_name = "technical_lifetime"
    map(
        eachrow(filter(r_ins -> r_ins[Symbol("instance")] == instance, df_diff_asset))
    ) do df_row
        TIO.update_tbl(
            _connection, TIO_tbl,
            Dict(param_name => df_row[Symbol(param_name)]),
            where_ = "asset = '$(df_row.asset)'"
        )
    end
end;

# TIO.get_table(_connection, "asset_both")

4-element Vector{Nothing}:
 nothing
 nothing
 nothing
 nothing

In [27]:
# Run the model instance
multiyear_vintage_standard = TEM.run_scenario(
    connection_vintage_standard;
    optimizer_parameters = Dict("output_flag" => true), # TEM.default_parameters(HiGHS.Optimizer): Dict("output_flag" => false)
    output_folder = output_dir, 
    # model_file_name = joinpath(output_dir, "model.lp"),
    show_log=true,
    # # To record solving time. 
    # # Always run with this option in a new or restarted kernel to ensure the time is dedicated to this solve only.
    # log_file = joinpath(@__DIR__, "model-instance-Tulipa/model-log_$(instance).txt")
)

Running HiGHS 1.15.0 (git hash: 8396001901): Copyright (c) 2026 under MIT licence terms
Includes third-party software components, see THIRD_PARTY_NOTICES.md for full details
MIP has 420480 rows; 788406 cols; 1664382 nonzeros; 6 integer variables (0 binary)
Coefficient ranges:
  Matrix  [3e-06, 1e+00]
  Cost    [1e-03, 3e+03]
  Bound   [1e+02, 2e+02]
  RHS     [4e-04, 9e+01]
Presolving model
236511 rows, 630672 cols, 1191234 nonzeros 2s
228431 rows, 611278 cols, 1157117 nonzeros 4s
Presolve reductions: rows 228431(-192049); columns 611278(-177128); nonzeros 1157117(-507265) 

Solving MIP model with:
   228431 rows
   611278 cols (0 binary, 6 integer, 0 implied int., 611272 continuous, 0 domain fixed)
   1157117 nonzeros
   Thread count 4 (of 8 threads). Using 1 max workers. Parallel search off

Src: B => Branching; C => Central rounding; F => Feasibility pump; H => Heuristic;
     I => Shifting; J => Feasibility jump; L => Sub-MIP; P => Empty MIP; R => Randomized rounding;
     S => Sol

EnergyProblem:
  - Model created!
    - Number of variables: 788406
    - Number of constraints for variable bounds: 788406
    - Number of structural constraints: 420480
  - Model solved!
    - Termination status: OPTIMAL
    - Objective value: 771132.5967944773
    - Objective breakdown:
      - assets_fixed_cost_aggregated_vintage_method: 152488.5502744505
      - assets_fixed_cost_compact_vintage_method: 0.0
      - assets_investment_cost: 291102.32146275806
      - flows_fixed_cost: 0.0
      - flows_investment_cost: 0.0
      - flows_operational_cost: 327541.72505726886
      - storage_assets_energy_fixed_cost: 0.0
      - storage_assets_energy_investment_cost: 0.0
      - units_on_operational_cost: 0.0
      - vintage_flows_operational_cost: 0.0


### 2.2 Key results

#### Capacity

- **Assumption**:
    - assets of a vintage is exclusively investable in the same milestone year as the vintage indicated by the asset name: see parameter `investable` in `asset-milestone.csv`.

In [28]:
print_capacity_investment(row -> occursin("wind", row.asset), connection_vintage_standard, DECOMM)

Initial wind capacity (GW): 3×5 DataFrame
 Row │ asset   milestone_year  commission_year  decommissionable  initial_units 
     │ String  Int32           Int32            Bool              Float64       
─────┼──────────────────────────────────────────────────────────────────────────
   1 │ wind25            2030             2030             false           30.0
   2 │ wind25            2040             2040             false           30.0
   3 │ wind25            2050             2050             false           30.0

Invested wind capacity (GW): # of rows = # of investment variables.
3×6 DataFrame
 Row │ asset   milestone_year  investment_integer  capacity  investment_limit  solution 
     │ String  Int32           Bool                Float64   Float64           Float64  
─────┼──────────────────────────────────────────────────────────────────────────────────
   1 │ wind30            2030                true       1.0           107.567     107.0
   2 │ wind40            2040        

#### Annual productions & total system cost

In [29]:
print_annual_total_prod(connection_vintage_standard, 2030, 2040, 2050)

total_cost = multiyear_vintage_standard.objective_value
println("Total system cost: $(round(total_cost/1000, digits=2)) Billion €")

inv_cost, fixed_om_cost, variable_om_cost = objective_terms_value(multiyear_vintage_standard, connection_vintage_standard)
println(
    "\t investment: $(round(inv_cost/1000, digits=2)) Billion € \n", 
    "\t fixed O&M: $(round(fixed_om_cost/1000, digits=2)) Billion € \n",
    "\t variable O&M: $(round(variable_om_cost/1000, digits=2)) Billion €"
)
println(
    "Total system cost ≈ investment + fixed O&M + variable O&M: ", 
    total_cost ≈ inv_cost + fixed_om_cost + variable_om_cost
)
# @assert total_cost ≈ inv_cost + fixed_om_cost + variable_om_cost

2030s
	 wind prodution: 611.93 TWh p.a.
	 market supply: 200.73 TWh p.a.
2040s
	 wind prodution: 954.75 TWh p.a.
	 market supply: 174.69 TWh p.a.
2050s
	 wind prodution: 1025.13 TWh p.a.
	 market supply: 205.1 TWh p.a.
Total system cost: 771.13 Billion €
	 investment: 291.1 Billion € 
	 fixed O&M: 152.49 Billion € 
	 variable O&M: 327.54 Billion €
Total system cost ≈ investment + fixed O&M + variable O&M: true


## Instance 3. Explicit technology vintage of units - compact method

- Difference from **Instance 2**: `asset.csv` (only tha parameter `vintage_method`), `asset-commission.csv`, `asset-both`
- In any milestone year, only the tech. vintage of the same year is available for investment (assumed with setting `vintage_method=compact_profiles` for "wind" `asset`)
- Resulted investment in 2050 differ significantly from the no vintage case because the wind of 2020 vintage (wind commissioned in 2020, defined in `asset-commission.csv`) uses the default availability of `1.0` over the milestone year 2050 due to the lack of availability value is given for 2050 for this vintage (`availability-wind2020` records in `profiles-rep-periods.csv`, which is assigned to the wind commissioned in 2020 in `assets-profiles.csv`)

### 3.1 Build and run the model instance

In [30]:
instance = "3-vintage-compact"
# Define and build the input output directories
input_dir = "model-instance-Tulipa/input-$(instance)"
output_dir = joinpath(@__DIR__, "model-instance-Tulipa/output-$(instance)")

# Always build a new result directory
rm(output_dir, force=true, recursive=true) 
mkdir(output_dir);

# Create the connection and prepare input data
connection_vintage_compact = DBInterface.connect(DuckDB.DB)
TIO.read_csv_folder(connection_vintage_compact, input_dir)
TEM.populate_with_defaults!(connection_vintage_compact)

#### 3.1a (Optional) Config decommissioning and technical lifetime

- see `./config-instance/decommission/asset-both_decommission.csv` for value setup
- **Assumption:**
    - assets that are not investable (i.e. 2025 wind vintage) should not be decommissionable either
    - assets should not be decommissionable before and in its commission (vintage) year


In [35]:
DECOMM && begin
    _connection = connection_vintage_compact

    TIO_tbl = "asset_both"
    TIO_tbl_column = "commission_year"  # compact metthod differentiates asset vintages by commission_year
    param_name = "decommissionable"
    map(
        eachrow(filter(r_ins -> r_ins[Symbol("instance")] == instance, df_diff_asset_both))
    ) do df_row
        TIO.update_tbl(
            _connection, TIO_tbl,
            Dict(param_name => df_row[Symbol(param_name)]),
            where_ = "asset = '$(df_row.asset)' AND $(TIO_tbl_column) = '$(df_row[Symbol(TIO_tbl_column)])'"
        )
    end

    TIO_tbl = "asset"
    TIO_tbl_column = "asset"
    param_name = "technical_lifetime"
    map(
        eachrow(filter(r_ins -> r_ins[Symbol("instance")] == instance, df_diff_asset))
    ) do df_row
        TIO.update_tbl(
            _connection, TIO_tbl,
            Dict(param_name => df_row[Symbol(param_name)]),
            where_ = "asset = '$(df_row.asset)'"
        )
    end
end;

# TIO.get_table(_connection, "asset_both")

1-element Vector{Nothing}:
 nothing

In [36]:
# Run the model instance
multiyear_vintage_compact = TEM.run_scenario(
    connection_vintage_compact;
    optimizer_parameters = Dict("output_flag" => true), # TEM.default_parameters(HiGHS.Optimizer): Dict("output_flag" => false)
    output_folder = output_dir, 
    # model_file_name = joinpath(output_dir, "model.lp"),
    show_log=true,
    # # To record solving time. 
    # # Always run with this option in a new or restarted kernel to ensure the time is dedicated to this solve only.
    # log_file = joinpath(@__DIR__, "model-instance-Tulipa/model-log_$(instance).txt")
)

Running HiGHS 1.15.0 (git hash: 8396001901): Copyright (c) 2026 under MIT licence terms
Includes third-party software components, see THIRD_PARTY_NOTICES.md for full details
MIP has 341645 rows; 315368 cols; 762110 nonzeros; 8 integer variables (0 binary)
Coefficient ranges:
  Matrix  [3e-06, 1e+00]
  Cost    [1e-03, 3e+03]
  Bound   [1e+02, 2e+02]
  RHS     [4e-04, 9e+01]
Presolving model
26285 rows, 149310 cols, 280692 nonzeros 0s
25077 rows, 132476 cols, 218815 nonzeros 2s
Presolve reductions: rows 25077(-316568); columns 132476(-182892); nonzeros 218815(-543295) 

Solving MIP model with:
   25077 rows
   132476 cols (0 binary, 6 integer, 0 implied int., 132470 continuous, 0 domain fixed)
   218815 nonzeros
   Thread count 4 (of 8 threads). Using 1 max workers. Parallel search off

Src: B => Branching; C => Central rounding; F => Feasibility pump; H => Heuristic;
     I => Shifting; J => Feasibility jump; L => Sub-MIP; P => Empty MIP; R => Randomized rounding;
     S => Solve LP; T 

EnergyProblem:
  - Model created!
    - Number of variables: 315368
    - Number of constraints for variable bounds: 315368
    - Number of structural constraints: 341645
  - Model solved!
    - Termination status: OPTIMAL
    - Objective value: 771132.5967944773
    - Objective breakdown:
      - assets_fixed_cost_aggregated_vintage_method: 0.0
      - assets_fixed_cost_compact_vintage_method: 152488.5502744505
      - assets_investment_cost: 291102.32146275806
      - flows_fixed_cost: 0.0
      - flows_investment_cost: 0.0
      - flows_operational_cost: 327541.7250572685
      - storage_assets_energy_fixed_cost: 0.0
      - storage_assets_energy_investment_cost: 0.0
      - units_on_operational_cost: 0.0
      - vintage_flows_operational_cost: 0.0


### 3.2 Key results

#### Capacity

- **Assumption**:
    - assets of a vintage is exclusively investable in the same milestone year as the commission (vintage) year: see parameter `investable` in `asset-milestone.csv`.

- Units built in different years are explicitly listed, meaning that their corresponding profiles are also considered.
- *`Open question`* Number of decommissioning variables is subject to the assumption on decommissionable milestone years. This seems unfair to compare with the investment variable which is limited to the same milestone year of its commision year (vintage).

In [33]:
print_capacity_investment(row -> occursin("wind", row.asset), connection_vintage_compact, DECOMM)

Initial wind capacity (GW): 3×5 DataFrame
 Row │ asset   milestone_year  commission_year  decommissionable  initial_units 
     │ String  Int32           Int32            Bool              Float64       
─────┼──────────────────────────────────────────────────────────────────────────
   1 │ wind              2030             2025             false           30.0
   2 │ wind              2040             2025             false           30.0
   3 │ wind              2050             2025             false           30.0

Invested wind capacity (GW): # of rows = # of investment variables.
3×6 DataFrame
 Row │ asset   milestone_year  investment_integer  capacity  investment_limit  solution 
     │ String  Int32           Bool                Float64   Float64           Float64  
─────┼──────────────────────────────────────────────────────────────────────────────────
   1 │ wind              2030                true       1.0           107.567     107.0
   2 │ wind              2040        

#### Annual productions & total system cost

In [34]:
print_annual_total_prod(connection_vintage_compact, 2030, 2040, 2050)

total_cost = multiyear_vintage_compact.objective_value
println("Total system cost: $(round(total_cost/1000, digits=2)) Billion €")

inv_cost, fixed_om_cost, variable_om_cost = objective_terms_value(multiyear_vintage_compact, connection_vintage_compact)
println(
    "\t investment: $(round(inv_cost/1000, digits=2)) Billion € \n", 
    "\t fixed O&M: $(round(fixed_om_cost/1000, digits=2)) Billion € \n",
    "\t variable O&M: $(round(variable_om_cost/1000, digits=2)) Billion €"
)
println(
    "Total system cost ≈ investment + fixed O&M + variable O&M: ", 
    total_cost ≈ inv_cost + fixed_om_cost + variable_om_cost
)
# @assert total_cost ≈ inv_cost + fixed_om_cost + variable_om_cost

2030s
	 wind prodution: 611.93 TWh p.a.
	 market supply: 200.73 TWh p.a.
2040s
	 wind prodution: 954.75 TWh p.a.
	 market supply: 174.69 TWh p.a.
2050s
	 wind prodution: 1025.13 TWh p.a.
	 market supply: 205.1 TWh p.a.
Total system cost: 771.13 Billion €
	 investment: 291.1 Billion € 
	 fixed O&M: 152.49 Billion € 
	 variable O&M: 327.54 Billion €
Total system cost ≈ investment + fixed O&M + variable O&M: true
